In [27]:
import pandas as pd
import numpy as np
import requests

## Прочитай JSON-файл, сохранённый в ex02.
- Одна из колонок имеет тип float, задай формат отображения через pd.options.display.float_format: числа с плавающей точкой должны показываться с двумя знаками после запятой.
- В столбце Model есть пропущенные значения; ничего с ними не делай.

In [28]:
pd.options.display.float_format = '{:.2f}'.format
#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)

In [29]:
df = pd.read_json('../data/auto.json', orient='records')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 725 entries, 0 to 724
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  725 non-null    str    
 1   Refund     725 non-null    int64  
 2   Fines      725 non-null    float64
 3   Make       725 non-null    str    
 4   Model      716 non-null    str    
dtypes: float64(1), int64(1), str(3)
memory usage: 28.4 KB


In [30]:
df.head()

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.00,Ford,Focus
1,E432XX77RUS,1,6500.00,Toyota,Camry
2,7184TT36RUS,1,2100.00,Ford,Focus
3,X582HE161RUS,2,2000.00,Ford,Focus
4,92918M178RUS,1,5700.00,Ford,Focus


## Обогати DataFrame сэмплом из этого же DataFrame.
- Создай сэмпл из 200 новых наблюдений, используя random_state = 21.
- В сэмпле не должно появиться новых комбинаций car number, make и model, чтобы набор данных оставался согласованным.
- На столбцы refund и fines ограничений нет: можно случайно выбирать их значения и назначать любому номеру автомобиля.
- Конкатенируй сэмпл с исходным DataFrame и получи новый DataFrame concat_rows.

In [31]:
RANDOM_STATE = 21
np.random.seed(RANDOM_STATE)
SAMPLE_SIZE = 200

In [32]:
sample_cars = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE, replace=True)
sample_cars['Refund'] = np.random.randint(df['Refund'].min(), df['Refund'].max() + 1, SAMPLE_SIZE)
sample_cars['Fines'] = np.random.randint(df['Fines'].min(), df['Fines'].max() + 1, SAMPLE_SIZE)
sample_cars = sample_cars.reset_index(drop=True)

concat_rows = pd.concat([df, sample_cars], ignore_index=True)
concat_rows.info()

<class 'pandas.DataFrame'>
RangeIndex: 925 entries, 0 to 924
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  925 non-null    str    
 1   Refund     925 non-null    int64  
 2   Fines      925 non-null    float64
 3   Make       925 non-null    str    
 4   Model      914 non-null    str    
dtypes: float64(1), int64(1), str(3)
memory usage: 36.3 KB


In [33]:
print(f'В сэмпл попало {sample_cars['Model'].isna().sum()} машины без указанной модели')

В сэмпл попало 2 машины без указанной модели


## Обогати concat_rows новой колонкой с сгенерированными данными.
- Создай Series с именем Year со случайными целыми числами от 1980 до 2019.
- Перед генерацией лет вызови np.random.seed(21).
- Конкатенируй Series с DataFrame и назови получившийся DataFrame fines.

In [34]:
np.random.seed(RANDOM_STATE)

year = pd.Series(np.random.randint(1980, 2020, len(concat_rows)), name='Year')

fines = pd.concat([concat_rows, year], axis=1)
fines.info()

<class 'pandas.DataFrame'>
RangeIndex: 925 entries, 0 to 924
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  925 non-null    str    
 1   Refund     925 non-null    int64  
 2   Fines      925 non-null    float64
 3   Make       925 non-null    str    
 4   Model      914 non-null    str    
 5   Year       925 non-null    int64  
dtypes: float64(1), int64(2), str(3)
memory usage: 43.5 KB


## Обогати DataFrame данными из другого DataFrame.
- Создай новый DataFrame с номерами автомобилей и их владельцами.
- Возьми самые популярные фамилии в США (файл surname.json в папке datasets).
- Создай новую Series с фамилиями: без спецсимволов (запятых, скобок и т. п.); количество должно совпадать с числом уникальных номеров автомобилей в сэмпле (используй random_state = 21).
- Создай DataFrame owners с колонками CarNumber и SURNAME.

In [35]:
data = pd.read_json('../../datasets/surname.json')
data.columns = data.iloc[0]
data = data.iloc[1:].reset_index(drop=True)
surnames = data['NAME'].tolist()
surnames[1:5]

['ALLEN', 'ALVAREZ', 'ANDERSON', 'BAILEY']

In [36]:
np.random.seed(RANDOM_STATE)

unique_cars = fines['CarNumber'].unique()
surname = pd.Series(
    np.random.choice(surnames, len(unique_cars), replace=True),
    name='SURNAME'
)
surname.head()

0    RICHARDSON
1          ROSS
2        MORGAN
3        BAILEY
4         LOPEZ
Name: SURNAME, dtype: str

In [37]:
owners = pd.DataFrame({
    'CarNumber': unique_cars,
    'SURNAME': surname.values
})
owners.head()

,CarNumber,SURNAME
0,Y163O8161RUS,RICHARDSON
1,E432XX77RUS,ROSS
2,7184TT36RUS,MORGAN
3,X582HE161RUS,BAILEY
4,92918M178RUS,LOPEZ


In [38]:
owners.info()

<class 'pandas.DataFrame'>
RangeIndex: 531 entries, 0 to 530
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   CarNumber  531 non-null    str  
 1   SURNAME    531 non-null    str  
dtypes: str(2)
memory usage: 8.4 KB


## Добавь ещё пять наблюдений в fines (придумай свои CarNumber и т. п.).

In [39]:
new_cars = pd.DataFrame({
    'CarNumber': ['A11111111RUS', 'B222BB22RUS', '3333CC33RUS', 'P666DD067RUS', '12345X678RUS'],
    'Refund': [1, 2, 2, 2, 1],
    'Fines': [450.00, 1984.00, 5200.00, 1234.00, 4267.00],
    'Make': ['Toyota', 'Skoda', 'Ford', 'Ford', 'Toyota'],
    'Model': ['Camry', 'Octavia', 'Focus', 'Focus', 'Camry'],
    'Year': [2019, 2018, 2017, 2016, 2015]
})

fines = pd.concat([fines, new_cars], ignore_index=True)
fines.tail(10)

,CarNumber,Refund,Fines,Make,Model,Year
920,M942OT152RUS,1,153748.00,Ford,Focus,1981
921,Y187O8161RUS,1,139009.00,Ford,Focus,1992
922,7064C8197RUS,2,21327.00,Volkswagen,Passat,2007
923,8437XX154RUS,2,110381.00,Ford,Focus,2005
924,C410X938RUS,1,81646.00,Ford,Focus,1997
925,A11111111RUS,1,450.00,Toyota,Camry,2019
926,B222BB22RUS,2,1984.00,Skoda,Octavia,2018
927,3333CC33RUS,2,5200.00,Ford,Focus,2017
928,P666DD067RUS,2,1234.00,Ford,Focus,2016
929,12345X678RUS,1,4267.00,Toyota,Camry,2015


In [40]:
fines.info()

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  930 non-null    str    
 1   Refund     930 non-null    int64  
 2   Fines      930 non-null    float64
 3   Make       930 non-null    str    
 4   Model      919 non-null    str    
 5   Year       930 non-null    int64  
dtypes: float64(1), int64(2), str(3)
memory usage: 43.7 KB


## Удали последние 20 наблюдений из owners и добавь три новых, отличных от добавленных в fines.

In [41]:
owners = owners.iloc[:-20]
owners.info()

<class 'pandas.DataFrame'>
RangeIndex: 511 entries, 0 to 510
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   CarNumber  511 non-null    str  
 1   SURNAME    511 non-null    str  
dtypes: str(2)
memory usage: 8.1 KB


In [42]:
new_owners = pd.DataFrame({
    'CarNumber': ['9999WW999RUS', 'L777LL77RUS', '2810NN02RUS'],
    'SURNAME': ['SOBACHKIN', 'KOSHECHKIN', 'KRAVCHUK']
})

owners = pd.concat([owners, new_owners], ignore_index=True)
owners.tail()

,CarNumber,SURNAME
509,O50197197RUS,WRIGHT
510,7608EE777RUS,HILL
511,9999WW999RUS,SOBACHKIN
512,L777LL77RUS,KOSHECHKIN
513,2810NN02RUS,KRAVCHUK


## Соедини два DataFrame.
- Новый DataFrame должен содержать только те номера, которые есть в обоих DataFrame.
- Новый DataFrame должен содержать все номера из обоих DataFrame.
- Новый DataFrame должен содержать только номера из fines.
- Новый DataFrame должен содержать только номера из owners.

In [43]:
df_inner = pd.merge(fines, owners, on='CarNumber', how='inner')
print('Машины, которые есть в обоих датафреймах')
print(f'Размерность: {df_inner.shape}')
df_inner.head()

Машины, которые есть в обоих датафреймах
Размерность: (903, 7)


,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989,RICHARDSON
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995,ROSS
2,7184TT36RUS,1,2100.00,Ford,Focus,1984,MORGAN
3,X582HE161RUS,2,2000.00,Ford,Focus,2015,BAILEY
4,92918M178RUS,1,5700.00,Ford,Focus,2014,LOPEZ


In [44]:
df_outer = pd.merge(fines, owners, on='CarNumber', how='outer')
print('Машины из обоих датафреймов')
print(f'Размерность: {df_outer.shape}')
df_outer.head()

Машины из обоих датафреймов
Размерность: (933, 7)


,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,12345X678RUS,1.00,4267.00,Toyota,Camry,2015.00,NaN
1,2810NN02RUS,NaN,NaN,NaN,NaN,NaN,KRAVCHUK
2,3333CC33RUS,2.00,5200.00,Ford,Focus,2017.00,NaN
3,704687163RUS,2.00,1400.00,Ford,Focus,2004.00,ADAMS
4,704787163RUS,2.00,2800.00,Ford,Focus,1992.00,MORGAN


In [45]:
df_left = pd.merge(fines, owners, on='CarNumber', how='left')
print('Машины, которые есть в fines')
print(f'Размерность: {df_left.shape}')
df_left.head()

Машины, которые есть в fines
Размерность: (930, 7)


,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989,RICHARDSON
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995,ROSS
2,7184TT36RUS,1,2100.00,Ford,Focus,1984,MORGAN
3,X582HE161RUS,2,2000.00,Ford,Focus,2015,BAILEY
4,92918M178RUS,1,5700.00,Ford,Focus,2014,LOPEZ


In [46]:
df_right = pd.merge(fines, owners, on='CarNumber', how='right')
print('Машины, которые есть в owners')
print(f'Размерность: {df_right.shape}')
df_right.head()

Машины, которые есть в owners
Размерность: (906, 7)


,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2.00,3200.00,Ford,Focus,1989.00,RICHARDSON
1,Y163O8161RUS,2.00,1600.00,Ford,Focus,1980.00,RICHARDSON
2,Y163O8161RUS,2.00,82942.00,Ford,Focus,2019.00,RICHARDSON
3,Y163O8161RUS,1.00,12931.00,Ford,Focus,2017.00,RICHARDSON
4,Y163O8161RUS,1.00,9666.00,Ford,Focus,2017.00,RICHARDSON


## Создай сводную таблицу (pivot table) из fines.
- Она должна выглядеть как на примере ниже (значения - суммы штрафов), но со всеми годами. Твои значения могут отличаться.

In [47]:
pivot = pd.pivot_table(
    fines,
    values='Fines',
    index=['Make', 'Model'],
    columns='Year',
    aggfunc='sum'
)
pivot

Year                    1980      1981      1982      1983      1984  \
Make       Model                                                       
Ford       Focus   263540.59 868772.17 721008.76 507350.00 320398.59   
           Mondeo        NaN       NaN       NaN       NaN       NaN   
Skoda      Octavia 246755.00       NaN   6900.00  11594.59   4008.00   
Toyota     Camry   138226.00   8594.59       NaN   7200.00       NaN   
           Corolla       NaN       NaN   2000.00       NaN       NaN   
Volkswagen Golf     30900.00       NaN       NaN   8594.59    300.00   
           Jetta         NaN       NaN       NaN       NaN       NaN   
           Passat        NaN 102794.00       NaN   3200.00  10000.00   
           Touareg       NaN       NaN       NaN       NaN       NaN   

Year                    1985      1986      1987      1988      1989  ...  \
Make       Model                                                      ...   
Ford       Focus   366789.76 131437.59 617392.00 497048.59 374376.00  ...   
           Mondeo        NaN       NaN       NaN       NaN   8600.00  ...   
Skoda      Octavia  10294.59    600.00 284351.00       NaN  91400.00  ...   
Toyota     Camry         NaN       NaN       NaN       NaN  22400.00  ...   
           Corolla       NaN  72933.00   8000.00       NaN   4000.00  ...   
Volkswagen Golf     24000.00       NaN   9300.00       NaN 167630.00  ...   
           Jetta         NaN       NaN       NaN       NaN       NaN  ...   
           Passat    5000.00  15000.00  12300.00       NaN       NaN  ...   
           Touareg   5800.00       NaN       NaN       NaN       NaN  ...   

Year                    2010      2011      2012      2013      2014  \
Make       Model                                                       
Ford       Focus   576504.17 214020.17  97611.00 707743.59 488292.59   
           Mondeo        NaN       NaN  34400.00       NaN       NaN   
Skoda      Octavia   3100.00    500.00    500.00 121392.59 173062.00   
Toyota     Camry         NaN       NaN  34821.59       NaN       NaN   
           Corolla  24000.00   8594.59       NaN       NaN       NaN   
Volkswagen Golf          NaN 133641.00       NaN  72576.00       NaN   
           Jetta         NaN       NaN       NaN       NaN       NaN   
           Passat    2800.00  12702.00 177904.00  78455.00       NaN   
           Touareg   6300.00       NaN       NaN       NaN   1300.00   

Year                    2015      2016      2017      2018      2019  
Make       Model                                                      
Ford       Focus   691007.00 503722.59 525014.00 684846.59 235816.00  
           Mondeo        NaN  46200.00       NaN       NaN       NaN  
Skoda      Octavia  46394.59    300.00       NaN 158184.00   9500.00  
Toyota     Camry     4267.00  65409.00  73061.00  13000.00  18550.00  
           Corolla       NaN       NaN   9600.00       NaN 179198.00  
Volkswagen Golf      2300.00       NaN       NaN       NaN       NaN  
           Jetta         NaN       NaN       NaN       NaN       NaN  
           Passat   51365.00   2100.00       NaN       NaN       NaN  
           Touareg    500.00       NaN       NaN       NaN       NaN  

[9 rows x 40 columns]

## Сохрани оба DataFrame - fines и owners - в CSV-файлы без индекса.

In [48]:
fines.to_csv('../data/fines.csv', index=False)
owners.to_csv('../data/owners.csv', index=False)

In [23]:
concat_rows.count()

CarNumber    925
Refund       925
Fines        925
Make         925
Model        914
dtype: int64

In [24]:
concat_rows.tail()

,CarNumber,Refund,Fines,Make,Model
920,M942OT152RUS,1,153748.00,Ford,Focus
921,Y187O8161RUS,1,139009.00,Ford,Focus
922,7064C8197RUS,2,21327.00,Volkswagen,Passat
923,8437XX154RUS,2,110381.00,Ford,Focus
924,C410X938RUS,1,81646.00,Ford,Focus


In [25]:
owners.head()

,CarNumber,SURNAME
0,Y163O8161RUS,RICHARDSON
1,E432XX77RUS,ROSS
2,7184TT36RUS,MORGAN
3,X582HE161RUS,BAILEY
4,92918M178RUS,LOPEZ


In [26]:
len(owners)

514